# Lab 05 (cluster): Evaluate AI systems and choose a model

**Chapter 4, "Evaluate AI Systems"** (sections *Evaluation Criteria*, *Model Selection*, *Design Your Evaluation Pipeline*)
**Environment:** Docker JupyterLab on the KU VPN | **Time:** Part A about 2.5 hours (graded). Part B is an optional stretch for bonus points.
**This is a Part-A-only week** because Side Project 1 is due. Do Part B only if you have time.

Larkspur has to pick **one** of the four course chat models for the help-desk assistant. Chapter 4 says not to pick from a leaderboard: decide what matters for *your* application, build an evaluation pipeline on *your* data, and let the numbers decide. That is this lab:

1. design evaluation criteria for the assistant,
2. run all four models on 40 **gold questions** (questions with a checked, correct reference answer) with the documents provided,
3. measure correctness, abstention, latency, and output length,
4. spot-check the LLM judge by reading some of its verdicts yourself,
5. dig into the **trap questions**, where the context contains two documents that disagree,
6. build a weighted scorecard and choose a model,
7. estimate what each model would cost per month through a (pretend) paid API, and where self-hosting becomes cheaper.

**Before you start**
- VPN on, Docker lab running.
- The main run sends about **290 requests** and takes roughly 20-30 minutes (the dense model is slow). Everything is cached in `lab05_cache.json`, so if your connection drops you can just re-run; finished requests are not repeated.

## Words you'll meet in this lab

You don't need to memorize these. Each one is explained again where it first matters; come back here if a word stops you.

| Term | What it means | Why it matters |
|---|---|---|
| **Evaluation set, gold questions** | A fixed list of questions, each with a checked, correct **reference answer**. "Gold" means the reference is trusted as the truth. | Every model answers the same questions, so the comparison is fair. |
| **Context** | The documents placed in the prompt next to the question (from Lab 01: everything the model reads for one answer). | The model should answer only from these documents. |
| **Trap question** | A question whose context contains two documents that disagree, for example an old and a current version of a policy. | Real company documents go out of date; the assistant must pick the current one. |
| **Unanswerable question, abstention** | A question whose answer is not in the context. **Abstaining** means the model declines ("Not in the documents") instead of guessing. | A help-desk assistant that makes things up is worse than one that says "I don't know". |
| **Evaluation criteria, metric** | A criterion is something you care about (for example correctness or speed). A metric is the number you measure it with. | Deciding them before you see any results keeps the choice honest. |
| **LLM judge** | A model that grades another model's answers against the reference (from Lab 04). | It can judge answers that are worded differently from the reference, but it can be biased. |
| **Spot check** | A person reads a small sample of the judge's verdicts and records whether they agree. | It is how you find out whether an automated evaluation can be trusted before anyone acts on it. |
| **Exact checks, regex** | Checks that compare text directly, with no model involved. A **regex** (regular expression) is a pattern for finding text, such as "a dollar sign followed by digits". | Cheap and repeatable, so use them where they work. |
| **Latency, p50 / p95** | Latency is the seconds a request takes (from Lab 02). **p50** is the median request; **p95** is the time 95% of requests beat, so it shows the slow ones users sometimes hit. | A help desk notices the slow answers, not just the typical one. |
| **Input / output tokens** | Tokens sent to the model (instructions, documents, question) and tokens it writes back (from Lab 01). | Hosted APIs charge per token, so they drive cost. |
| **Temperature 0** | A sampling setting that makes the model pick its most likely token each time (from Lab 03). | Results are (nearly) repeatable, which an evaluation needs. |
| **Cache** | A file (`lab05_cache.json`) where each finished reply is saved. | If your connection drops, re-running skips the requests already done. |
| **Weighted scorecard** | Each metric is rescaled to 0-1 and multiplied by how much you said it matters (its **weight**); the results are added up. | Turns many numbers into one ranking, using your priorities. |
| **Benchmark, leaderboard** | Public test sets and the rankings of models on them. | Q3 asks why they are a weak guide for choosing a model for one specific application. |
| **Build vs. buy** | Hosting a model yourself versus paying a company's API per token. | A9 estimates the cost of each; optional Part B turns it into a recommendation. |
| **Break-even volume** | The number of questions per month at which the API bill equals the fixed cost of self-hosting. | Below it the API is cheaper; above it, self-hosting is. |

## Setup

Run both cells. The first connects to the course gateway (from Lab 02) and lists its models. The second defines `call()`, a helper that sends one request and saves the reply in the **cache** file `lab05_cache.json`, so a re-run never repeats a finished request. It then loads the gold questions and their documents from the corpus (files ending in `.jsonl`, "JSON Lines": one record per line) and sends each model a one-word test message. Each printed line is a model, its reply, and how many seconds it took.

In [ ]:
import json, re, time, math, random, hashlib
from collections import Counter

import numpy as np
import pandas as pd

from itec632 import get_client, MODELS, NO_THINK, strip_think
from itec632.grading import LabChecker
from itec632.corpus import corpus_dir

client = get_client()
ck = LabChecker("lab05", part="cluster")
CANDIDATES = {"qwen-moe": MODELS["main"], "qwen-dense": MODELS["dense"], "gemma-moe": MODELS["gemma"], "qwen-small": MODELS["small"]}
print("Gateway models:", [m.id for m in client.models.list().data])

In [ ]:
CACHE_FILE = "lab05_cache.json"
try:
    CACHE = json.load(open(CACHE_FILE))
except Exception:
    CACHE = {}
REQUESTS = {"n": 0}
MIN_GAP = 1.5
_last_call = [0.0]


def call(model, messages, max_tokens=200, schema=None, cache_key=None):
    """One chat request. Returns a dict: text, latency_s, out_tokens, in_tokens (cached by cache_key).

    Latency is wall-clock time for the whole (non-streamed) request, measured on the first run only.
    The cache key also includes a fingerprint of the prompt, so if a prompt changes, the request is sent again.
    """
    if cache_key:
        cache_key = f"{cache_key}|{hashlib.sha256(json.dumps([model, messages]).encode()).hexdigest()[:10]}"
    if cache_key and cache_key in CACHE:
        return CACHE[cache_key]
    kwargs = dict(model=model, messages=messages, max_tokens=max_tokens, temperature=0.0, extra_body=NO_THINK)
    if schema is not None:
        kwargs["response_format"] = {"type": "json_schema", "json_schema": {"name": "result", "schema": schema}}
    result = {"text": "", "latency_s": None, "out_tokens": None, "in_tokens": None}
    for attempt in range(3):
        wait = MIN_GAP - (time.time() - _last_call[0])
        if wait > 0:
            time.sleep(wait)
        try:
            _last_call[0] = time.time()
            REQUESTS["n"] += 1
            t0 = time.perf_counter()
            r = client.chat.completions.create(**kwargs)
            latency = time.perf_counter() - t0
            usage = getattr(r, "usage", None)
            result = {"text": strip_think(r.choices[0].message.content or ""), "latency_s": latency,
                      "out_tokens": getattr(usage, "completion_tokens", None),
                      "in_tokens": getattr(usage, "prompt_tokens", None)}
            break
        except Exception as e:
            msg = str(e)
            if "response_format" in kwargs and ("response_format" in msg or "schema" in msg or "400" in msg):
                kwargs.pop("response_format")
                continue
            print(f"   (attempt {attempt + 1} failed: {msg[:100]})")
            time.sleep(5 * (attempt + 1))
    if cache_key and result["text"]:
        CACHE[cache_key] = result
        json.dump(CACHE, open(CACHE_FILE, "w"))
    return result


EVAL = corpus_dir() / "eval"
GOLD = [json.loads(line) for line in open(EVAL / "gold_qa.jsonl")]
BY_ID = {q["qid"]: q for q in GOLD}
CTX = {}
for line in open(EVAL / "gold_contexts.jsonl"):
    rec = json.loads(line)
    CTX[rec["qid"]] = rec["passages"]


def is_unanswerable(q):
    return q["reference_answer"].startswith("Not in the documents")


def format_context(qid):
    parts = [f"[{p['doc_id']} v{p['version']}, effective {p['effective_date']}] {p['title']} - {p['section']}\n{p['text']}"
             for p in CTX.get(qid, [])]
    return "\n\n---\n\n".join(parts) if parts else "(no documents)"


for name, model in CANDIDATES.items():
    r = call(model, [{"role": "user", "content": "Reply with the single word: ready"}], max_tokens=10, cache_key=f"ping|{name}")
    print(f"{name:11} -> {r['text']!r}  ({r['latency_s'] if r['latency_s'] is None else round(r['latency_s'], 2)} s)")

# Part A (graded)

## A1. Design the evaluation criteria

Chapter 4 groups evaluation criteria into **domain-specific capability**, **generation capability** (for us mainly factual consistency with the documents), **instruction-following capability**, and **cost and latency**. In plain words, for this assistant:

- *domain-specific capability*: does it get help-desk questions right?
- *generation capability*: does what it writes stay true to the documents, without made-up details?
- *instruction-following capability*: does it do what the system prompt says (short answers, "Not in the documents" when it can't answer)?
- *cost and latency*: how much does each answer cost, and how long does it take?

Write down what each means *for the Larkspur help-desk assistant* before you see any results; deciding after you see the numbers is how people talk themselves into the model they already liked.

The pipeline below will produce these metrics for every model (pick from them):

| Metric | Meaning | Better | Fits criterion |
|---|---|---|---|
| `judge_correct` | LLM-judge correctness on all 32 answerable questions, traps included (correct = 1, partly = 0.5) | higher | domain capability |
| `fact_recall` | share of key facts from the reference (amounts, times, names, counts) that appear in the answer | higher | domain or generation capability |
| `trap_accuracy` | `judge_correct` on the 16 trap questions only (does it stay true to the *current* document?) | higher | generation capability |
| `abstention_rate` | share of unanswerable questions where the model declined | higher | instruction following (or generation: not making things up) |
| `false_abstention` | share of answerable questions where the model wrongly declined | lower | instruction following |
| `format_ok` | share of answers that follow the format (at most 2 sentences, no more than 80 words) | higher | instruction following |
| `latency_p50_s`, `latency_p95_s` | median / 95th percentile request time | lower | cost and latency |
| `out_tokens_mean` | mean output tokens per answer (a cost proxy) | lower | cost and latency |

A **percentile** comes from sorting all the request times: p50 is the middle one (the median) and p95 is the time that 95% of requests beat. A **proxy** is a stand-in measure: more output tokens means a higher bill. A share such as 0.75 means 75%.

**TODO:** fill in `CRITERIA`: for each of the four required criteria, a weight, **one** metric from the table, and one sentence on why it matters at Larkspur. Weights must sum to 1. A **weight** says how much a criterion counts: 0.40 means 40% of the final score.

The rules, spelled out: each criterion uses exactly one metric. If you want a second metric (say both latency and tokens), add a fifth criterion with its own name, weight, and metric. Note that `judge_correct` already includes the trap questions, so choosing both `judge_correct` and `trap_accuracy` counts the traps twice; that's allowed, but do it on purpose.

One example, so you can see the shape (you may choose differently): `"domain_capability": {"weight": 0.4, "metric": "judge_correct", "why": "A wrong rate or procedure costs a technician time and the client's trust."}`. There's no single right set of weights; Q1 asks you to defend yours.

In [ ]:
CRITERIA = {
    "domain_capability":     {"weight": None, "metric": "", "why": ""},   # TODO
    "generation_quality":    {"weight": None, "metric": "", "why": ""},   # TODO
    "instruction_following": {"weight": None, "metric": "", "why": ""},   # TODO
    "cost_latency":          {"weight": None, "metric": "", "why": ""},   # TODO
}

In [ ]:
METRICS = {"judge_correct": True, "fact_recall": True, "trap_accuracy": True, "abstention_rate": True,
           "false_abstention": False, "format_ok": True, "latency_p50_s": False, "latency_p95_s": False,
           "out_tokens_mean": False}          # metric -> higher_is_better
required = {"domain_capability", "generation_quality", "instruction_following", "cost_latency"}
weights_ok = all(isinstance(c.get("weight"), (int, float)) and c["weight"] >= 0 for c in CRITERIA.values())
ck.check("A1_criteria_complete", required <= set(CRITERIA) and weights_ok
         and all(c.get("metric") in METRICS and len(c.get("why", "")) >= 20 for c in CRITERIA.values()),
         points=2, hint="Every criterion needs a numeric weight, a metric from the METRICS table, and a 'why' sentence.")
ck.check("A1_weights_sum_to_1", weights_ok and abs(sum(c["weight"] for c in CRITERIA.values()) - 1) < 0.01,
         points=1, hint="Weights must add up to 1.0.")
ck.record("A1_weights", {k: c.get("weight") for k, c in CRITERIA.items()})

## A2. The evaluation set and its context

Chapter 4's pipeline advice: evaluate on data that looks like real use, include slices you care about, and keep the set fixed so models are compared on equal terms. Our 40 questions:

- **16 trap questions**: the gold context deliberately contains **two documents that disagree** (for example an old and a new version). The reference answer is the current truth. We are *not* telling the model or you which document is outdated.
- **16 ordinary answerable questions.**
- **8 unanswerable questions**: the context is related but does not contain the answer.

The 16 ordinary questions are picked at random, but with a fixed **seed** (the `5` in `random.Random(5)`), so everyone gets the same 16. Every model gets exactly the same system prompt and context. Look at one trap question's context below: what would *you* answer, and how did you decide? (In the Larkspur world, "today" is 2026-11-02.)

`format_context(qid)` works for **any** question: `print(format_context("Q012"))` prints its whole context. You'll need that for Q2.

In [ ]:
TRAP_IDS = ['Q063', 'Q019', 'Q030', 'Q057', 'Q047', 'Q079', 'Q058', 'Q006', 'Q028', 'Q011', 'Q039', 'Q022', 'Q062', 'Q035', 'Q078', 'Q021']
normal_pool = [q["qid"] for q in GOLD if not q["trap"] and q["category"] != "data" and not is_unanswerable(q)]
NORMAL_IDS = random.Random(5).sample(normal_pool, 16)
UNANS_IDS = [q["qid"] for q in GOLD if is_unanswerable(q)]
EVAL_IDS = TRAP_IDS + NORMAL_IDS + UNANS_IDS
print(len(EVAL_IDS), "questions;", Counter(BY_ID[i]["category"] for i in EVAL_IDS))

SYSTEM = ("You are the Larkspur Ridge IT help-desk assistant. Answer using ONLY the documents provided. "
          "Answer in one or two sentences with the key fact first. "
          "If the documents do not contain the answer, reply exactly: Not in the documents.")


def build_messages(qid):
    q = BY_ID[qid]
    return [{"role": "system", "content": SYSTEM},
            {"role": "user", "content": f"Documents:\n{format_context(qid)}\n\nQuestion: {q['question']}"}]


example = TRAP_IDS[1]
print("QUESTION:", BY_ID[example]["question"], "\n")
print(format_context(example)[:2500])

**Reading the output:** the first line counts the 40 questions by category. Then comes one trap question and its context. Each document starts with a header in square brackets: the document ID, its version number (`v...`), and its **effective date** (the date it took effect), followed by the title and section. Documents are separated by `---`.

The next cell lists every trap question with just the headers of its context documents, so you can see at a glance what kinds of documents disagree. Notice that the date is not the only clue: the **kind** of document matters too (a signed contract or price catalog, a technician runbook, a client FAQ). Keep this list in mind for Q2.

In [ ]:
for qid in TRAP_IDS:
    print(f"{qid}: {BY_ID[qid]['question']}")
    for p in CTX[qid]:
        print(f"     {p['doc_id']} v{p['version']} (effective {p['effective_date']}): {p['title'][:60]}")

## A3. Cheap, exact checks first: key facts, abstention, format

Before paying for an LLM judge, use exact checks where they work (Chapter 3's *exact evaluation*: comparing text directly, with no model involved). Three helpers:

- `key_facts(reference)` pulls the checkable facts out of a reference answer: dollar amounts, percentages, clock times, phone numbers, counts with units, hardware models, and staff names. (Given.)
- `fact_recall(answer, facts)`: the share of those facts that appear in the answer, from 0 (none) to 1 (all). It returns `None` when the reference has no checkable facts. (Given; you test where it goes wrong below.)
- `is_abstention(answer)`: **TODO**, does the answer decline? Models phrase it many ways ("not mentioned", "the documents don't specify"). Three patterns are given; add patterns until the test cases pass. Two more short patterns are enough.

`key_facts()` and `is_abstention()` use **regular expressions** (regex): small patterns for finding text. For example `\$\d+` means "a dollar sign followed by one or more digits", and `(?:does|do)` means "does or do". Look at the patterns already in `FACT_PATTERNS` and `ABSTAIN_PATTERNS` for more; an AI assistant is good at explaining a regex line by line.

In [ ]:
STAFF = ["Dana Whitfield", "Marcus Oyelaran", "Priya Raman", "Tom Kessler", "Renee Castillo", "Jordan Pike", "Alicia Brandt",
         "Kevin Duong", "Hannah Moreau", "Samuel Ortiz", "Lena Fischer", "Andre Mitchell", "Grace Liu", "Brian Novak",
         "Maya Patel", "Chris Bauer", "Tasha Greene", "Luis Herrera", "Emily Sandoval", "Noah Whitaker", "Olivia Grant",
         "Derek Simmons", "Rachel Kim", "Isaac Brennan", "Greg Halvorsen"]
FACT_PATTERNS = [
    r"\$\d[\d,]*(?:\.\d+)?",                         # $247.50
    r"\b\d+(?:\.\d+)?%",                              # 20%
    r"\b\d{1,2}:\d{2}\s?[AP]M\b",                      # 7:00 AM
    r"\(\d{3}\) \d{3}-\d{4}",                          # (913) 555-0142
    r"\b\d+(?:\.\d+)?\s(?:business\s)?(?:days?|hours?|minutes?|GB|years?|weeks?)\b",   # 5 business days
    r"\b(?:Latitude|OptiPlex(?: Micro)?)\s\d{4}\b",     # Latitude 5550
]


def _norm(s):
    return re.sub(r"\s+", " ", (s or "").lower().replace(",", "").replace("$", "")).strip()


def key_facts(reference):
    """Checkable facts in a reference answer (normalized strings). Empty list if none."""
    if reference.startswith("Not in the documents"):
        return []
    facts = []
    for pat in FACT_PATTERNS:
        facts += re.findall(pat, reference)
    facts += [n for n in STAFF if n in reference]
    out = []
    for f in facts:
        f = _norm(f).replace(".00", "")
        if f and f not in out:
            out.append(f)
    return out


for qid in EVAL_IDS[:4]:
    print(qid, key_facts(BY_ID[qid]["reference_answer"]), "<-", BY_ID[qid]["reference_answer"][:70])

**Reading the output:** each line is a question ID, the list of facts `key_facts()` found in its reference answer, and the start of that reference answer. Facts are lowercased with `$` and commas removed, so `$1,200` and `1200` count as the same fact. An empty list `[]` means there was nothing checkable.

In [ ]:
def fact_recall(answer, facts):
    """Share of `facts` (from key_facts) that appear in `answer`. None if there are no facts to check."""
    if not facts:
        return None
    a = _norm(answer).replace(".00", "")
    return sum(f in a for f in facts) / len(facts)       # facts found / facts to check


ABSTAIN_PATTERNS = [
    r"not in the (?:provided )?documents?",
    r"\bnot (?:mentioned|specified|stated|provided|included|listed|given|named|available)\b",
    r"\bno (?:information|mention|details?)\b",
    # TODO: add regex patterns (lowercase) so every abstention test below is caught and no non-abstention is
]


def is_abstention(answer):
    a = (answer or "").lower()
    return any(re.search(p, a) for p in ABSTAIN_PATTERNS)

In [ ]:
abstain_tests = {
    "Not in the documents.": True,
    "The documents do not specify the guest Wi-Fi password.": True,
    "That information is not mentioned in the provided documents.": True,
    "I can't find a 401(k) match in these documents.": True,
    "The documents don't say who replaced Greg.": True,
    "Essentials P2 response is 2 business hours.": False,
    "Call PagerLine at (913) 555-0142; it pages the on-call engineer.": False,
    "No. SMS codes are not allowed for new enrollments.": False,
}
abstain_ok = all(is_abstention(t) == want for t, want in abstain_tests.items())
for t, want in abstain_tests.items():
    if is_abstention(t) != want:
        print("WRONG:", repr(t), "expected", want)
ck.check("A3_is_abstention", abstain_ok, points=2, hint="Add patterns to ABSTAIN_PATTERNS until every test case is classified correctly.")

### Where does key-fact recall go wrong?

`fact_recall()` is cheap and needs no model, so it is tempting to trust it. Below are four answers to the after-hours rate question. `FR_CORRECT` records what a person would say about each.

**TODO (before you run the table):** predict the `fact_recall` score of each answer: `0`, `0.5`, or `1`. Your prediction is not graded on being right.

In [ ]:
FR_REF = "$247.50 per hour, with a 1 hour minimum."
FR_FACTS = key_facts(FR_REF)
print("Facts checked:", FR_FACTS)
FR_ANSWERS = {
    "a": "After-hours work is billed at $247.50 per hour with a 1 hour minimum.",
    "b": "It costs $247.50 an hour, and you always pay for at least one hour.",
    "c": "It is $247.50 per hour, and there is no 1 hour minimum.",
    "d": "It is $210 per hour with a 1 hour minimum.",
}
FR_CORRECT = {"a": True, "b": True, "c": False, "d": False}   # a person's verdict

In [ ]:
FR_PREDICT = {"a": None, "b": None, "c": None, "d": None}   # TODO: your predicted fact_recall for each answer: 0, 0.5 or 1

In [ ]:
fr_table = pd.DataFrame([{"answer_id": k, "answer": a, "correct?": FR_CORRECT[k], "your prediction": FR_PREDICT.get(k),
                          "fact_recall": fact_recall(a, FR_FACTS)} for k, a in FR_ANSWERS.items()]).set_index("answer_id")
display(fr_table)

**Reading the table:** `correct?` is the person's verdict and `fact_recall` the metric's score. A score of 1 should mean "all key facts present", but that is not the same as "correct".

**TODO:** name the **wrong** answer that still gets a full `fact_recall` of 1, and the **correct** answer that gets less than 1. In Q4 you can use these failures when you judge how far to trust the numbers.

In [ ]:
FR_TOO_HIGH = ""    # TODO: letter of the WRONG answer that fact_recall scores 1
FR_TOO_LOW = ""     # TODO: letter of the CORRECT answer that fact_recall scores below 1

In [ ]:
fr_scores = {k: fact_recall(a, FR_FACTS) for k, a in FR_ANSWERS.items()}
too_high = {k for k in FR_ANSWERS if not FR_CORRECT[k] and fr_scores[k] == 1}
too_low = {k for k in FR_ANSWERS if FR_CORRECT[k] and fr_scores[k] < 1}
ck.check("A3_fact_predictions", all(FR_PREDICT.get(k) in (0, 0.5, 1) for k in FR_ANSWERS), points=1,
         hint="Set every value in FR_PREDICT to 0, 0.5 or 1 (your guess, made before looking at the table).")
ck.check("A3_fact_recall_limits", FR_TOO_HIGH in too_high and FR_TOO_LOW in too_low, points=1,
         hint="FR_TOO_HIGH: a row where correct? is False but fact_recall is 1. FR_TOO_LOW: a row where correct? is True but fact_recall is below 1.")
ck.record("A3_fact_predictions_right", sum(FR_PREDICT.get(k) == v for k, v in fr_scores.items()))

## A4. Run all four models

Each model answers all 40 questions (160 requests). We keep the text, the latency, and the token counts. **Do not change the prompt between models**; if you change it, re-run all four.

`format_ok` is computed here: at most 2 sentences and at most 80 words.

A word of caution about **latency**: the cluster is shared, so your times include waiting in line behind classmates' requests. A model can look slow one evening and fine the next morning. Also, with 40 requests per model, p95 is roughly your second-slowest request. Treat latency differences as rough, especially small ones.

In [ ]:
def count_sentences(text):
    return len([s for s in re.split(r"(?<=[.!?])\s+(?=[A-Z])", (text or "").strip()) if s])


RUNS = []
t_start = time.time()
for name, model in CANDIDATES.items():
    t0 = time.time()
    for qid in EVAL_IDS:
        r = call(model, build_messages(qid), max_tokens=200, cache_key=f"ans|{name}|{qid}")
        RUNS.append({"model": name, "qid": qid, **r})
    print(f"{name:11} done in {time.time() - t0:.0f}s")
runs = pd.DataFrame(RUNS)
q_meta = pd.DataFrame([{"qid": i, "trap": BY_ID[i]["trap"], "unanswerable": is_unanswerable(BY_ID[i]),
                        "category": BY_ID[i]["category"]} for i in EVAL_IDS])
runs = runs.merge(q_meta, on="qid")
runs["abstained"] = runs["text"].apply(is_abstention)
runs["format_ok"] = runs["text"].apply(lambda t: bool(t) and count_sentences(t) <= 2 and len(t.split()) <= 80)
runs["facts"] = runs["qid"].apply(lambda i: key_facts(BY_ID[i]["reference_answer"]))
runs["fact_recall"] = [fact_recall(t, f) for t, f in zip(runs["text"], runs["facts"])]
print(f"total {time.time() - t_start:.0f}s; empty answers:", int((runs["text"] == "").sum()))
runs[["model", "qid", "latency_s", "out_tokens", "abstained", "fact_recall", "text"]].head(8)

**Reading the table:** the first 8 of 160 rows, one per model and question. `latency_s` is the seconds that request took, `out_tokens` the length of the answer in tokens, `abstained` whether `is_abstention()` thinks the model declined, `fact_recall` the share of key facts it included (empty, shown as `NaN` or `None`, when the question has no checkable facts), and `text` is the answer itself. Replies come from the cache on a re-run, so the latencies are always from your first run.

In [ ]:
ck.check("A4_all_models_answered", len(runs) == 160 and (runs["text"] != "").sum() >= 150, points=3,
         hint="You need 4 models x 40 answers with (almost) no empty replies. Re-run the cell; finished calls are cached.")
ck.check("A4_latency_measured", runs["latency_s"].notna().sum() >= 150, points=1, hint="Latency should be recorded for each answer.")

## A5. LLM judge for correctness

Fact recall misses correct answers phrased differently ("two business days") and cannot judge procedures. So `qwen-moe` judges every answer to an **answerable** question against the reference (128 requests). It also flags answers that **contradict** the reference, that is, state a specific different fact; on trap questions that almost always means the model followed the other document.

You built a judge in Lab 04, so this one is **given** (prompt and parser): just read it and run it. Remember Lab 04's finding: `qwen-moe` judging `qwen-moe`'s own answers may be biased. That is why this section ends with a **human spot check**. `VERDICT_SCHEMA` is a **JSON schema**: a description of the exact shape the judge's reply must have (Lab 06 covers these). **Parsing** means turning the judge's text reply into Python values. Parsing is defensive: unparseable judgments become `None` and are counted, not silently dropped.

The first cell ends by testing `parse_verdict()` on five sample judge replies: each printed line is a reply, then the `(verdict, contradicts)` pair it extracted (`None` means it couldn't tell).

In [ ]:
VERDICT_SCHEMA = {"type": "object", "properties": {
    "verdict": {"type": "string", "enum": ["correct", "partially_correct", "incorrect", "abstained"]},
    "contradicts_reference": {"type": "boolean"},
    "reason": {"type": "string"}}, "required": ["verdict", "contradicts_reference", "reason"]}

JUDGE_PROMPT = """You are checking an answer from Larkspur Ridge IT's help-desk assistant.
The REFERENCE ANSWER is correct and current. Compare the ASSISTANT ANSWER to it.
- verdict: "correct" if it states the reference's key facts with nothing conflicting; "partially_correct" if the main fact is right
  but something important is missing; "incorrect" if the main fact is wrong or missing; "abstained" if it says it cannot answer.
- contradicts_reference: true if the answer states a specific fact (number, time, name, rule, or procedure step) that conflicts
  with the reference, otherwise false.

QUESTION: {question}
REFERENCE ANSWER: {reference}
ASSISTANT ANSWER: {answer}

Reply with JSON only: {"verdict": "...", "contradicts_reference": true or false, "reason": "<one sentence>"}"""
VERDICT_SCORE = {"correct": 1.0, "partially_correct": 0.5, "incorrect": 0.0, "abstained": 0.0}


def parse_verdict(text):
    """Return (verdict or None, contradicts True/False/None) from a judge reply, tolerating messy output."""
    if not text:
        return None, None
    verdict, contra = None, None
    m = re.search(r"\{.*\}", text, re.S)
    if m:
        try:
            d = json.loads(m.group(0))
            v = str(d.get("verdict", "")).strip().lower().replace(" ", "_")
            verdict = v if v in VERDICT_SCORE else None
            c = d.get("contradicts_reference")
            contra = c if isinstance(c, bool) else ({"true": True, "yes": True, "false": False, "no": False}.get(str(c).lower()))
        except Exception:
            pass
    low = text.lower()
    if verdict is None:
        for v in ["partially_correct", "partially correct", "incorrect", "abstained", "correct"]:
            if re.search(r"\b" + v + r"\b", low):
                verdict = v.replace(" ", "_")
                break
    if contra is None:
        mc = re.search(r"contradicts?_?(?:reference)?\W{0,5}(true|false|yes|no)", low)
        contra = None if not mc else mc.group(1) in ("true", "yes")
    return verdict, contra


for t in ['{"verdict": "correct", "contradicts_reference": false, "reason": "ok"}', "Verdict: INCORRECT. contradicts: yes",
          "partially correct", "", "no idea"]:
    print(repr(t)[:60], "->", parse_verdict(t))

In [ ]:
JUDGE = MODELS["main"]
judged = []
answerable_runs = runs[~runs["unanswerable"]]
t0 = time.time()
for _, row in answerable_runs.iterrows():
    q = BY_ID[row["qid"]]
    prompt = (JUDGE_PROMPT.replace("{question}", q["question"]).replace("{reference}", q["reference_answer"])
              .replace("{answer}", row["text"] or "(empty answer)"))
    r = call(JUDGE, [{"role": "user", "content": prompt}], max_tokens=150, schema=VERDICT_SCHEMA,
             cache_key=f"judge|{row['model']}|{row['qid']}")
    v, c = parse_verdict(r["text"])
    judged.append({"model": row["model"], "qid": row["qid"], "verdict": v, "contradicts": c,
                   "judge_score": VERDICT_SCORE.get(v) if v else None, "judge_raw": r["text"]})
judged = pd.DataFrame(judged)
runs = runs.drop(columns=[c for c in ["verdict", "contradicts", "judge_score", "judge_raw"] if c in runs.columns]).merge(
    judged, on=["model", "qid"], how="left")
print(f"judged {len(judged)} answers in {time.time() - t0:.0f}s; unparseable verdicts:", int(judged["verdict"].isna().sum()))

In [ ]:
ck.check("A5_judged", len(judged) == 128 and judged["verdict"].notna().sum() >= 100, points=2,
         hint="Expected 128 judgments (4 models x 32 answerable questions) and most of them parsed.")

### A5b. Human spot check: do you agree with the judge?

Before anyone acts on an automated evaluation, someone should read a sample of it. Chapter 4 says the same in *Select evaluation methods* (part of *Design Your Evaluation Pipeline*): use automatic metrics as much as possible, but don't be afraid to fall back on human evaluation. In Lab 04 you saw how a judge can be biased; now check *this* judge on *your* results.

The next cell picks 5 judged answers: up to 2 that the judge did **not** mark correct, at least 1 where `qwen-moe` judged its own answer, and the rest at random. For each case it prints the question, the reference answer, the model's answer, and the judge's verdict with its one-sentence reason.

**TODO:** read each case and decide: would *you* give the same verdict, using the definitions in `JUDGE_PROMPT`? Fill in `SPOT_CHECK` with `True` (agree) or `False` (disagree) and a short note (why, especially when you disagree). This takes about 10 minutes and is exactly what an IT team does before trusting an AI evaluation tool. Q4 uses your result.

In [ ]:
N_SPOT = 5


def judge_reason(text):
    """The judge's one-sentence reason (from its JSON), or the start of its raw reply."""
    m = re.search(r"\{.*\}", text or "", re.S)
    try:
        return str(json.loads(m.group(0)).get("reason", ""))[:300]
    except Exception:
        return (text or "")[:200]


pool = runs[~runs["unanswerable"] & runs["verdict"].notna()]
picked = pool[pool["verdict"] != "correct"].sample(frac=1, random_state=632).head(2)
own = pool[(pool["model"] == "qwen-moe") & ~pool.index.isin(picked.index)]
if not (picked["model"] == "qwen-moe").any() and len(own):
    picked = pd.concat([picked, own.sample(1, random_state=632)])
rest = pool[~pool.index.isin(picked.index)]
SPOT = pd.concat([picked, rest.sample(min(N_SPOT - len(picked), len(rest)), random_state=632)]).reset_index(drop=True)
for i, r in SPOT.iterrows():
    q = BY_ID[r["qid"]]
    print(f"=== Case {i + 1}: {r['model']} on {r['qid']} ===")
    print("QUESTION: ", q["question"])
    print("REFERENCE:", q["reference_answer"])
    print("ANSWER:   ", r["text"])
    print(f"JUDGE:     {r['verdict']} (contradicts reference: {r['contradicts']}) - {judge_reason(r['judge_raw'])}\n")

In [ ]:
SPOT_CHECK = {   # TODO: for each case above, agree = True or False, plus a short note saying why
    1: {"agree": None, "note": ""},
    2: {"agree": None, "note": ""},
    3: {"agree": None, "note": ""},
    4: {"agree": None, "note": ""},
    5: {"agree": None, "note": ""},
}

In [ ]:
labeled = {k: v for k, v in SPOT_CHECK.items() if isinstance(v.get("agree"), bool) and len(v.get("note", "")) >= 5}
need = min(N_SPOT, len(SPOT))
ck.check("A5_spot_check", len(SPOT) > 0 and len(labeled) >= need, points=2,
         hint=f"Label all {need} cases: agree = True or False (not None) and a short note for each.")
agree_rate = sum(v["agree"] for v in labeled.values()) / len(labeled) if labeled else None
print(f"You agreed with the judge on {sum(v['agree'] for v in labeled.values())} of {len(labeled)} cases.")
ck.record("A5_spot_check_n", len(labeled))
ck.record("A5_spot_check_agree_rate", agree_rate)

## A6. The comparison table

`summarize()` (given) turns one model's 40 runs into one row of the metrics from your A1 table. Most of them are a **share**: the average of a yes/no or 0-1 column over the right subset of questions. For example, `abstention_rate` is the share of the 8 **unanswerable** questions where the model declined, while `false_abstention` is the share of **answerable** questions where it wrongly declined. `trap_accuracy` is the average judge score on the 16 trap questions only. `safe_mean()` ignores missing values, and `pctl()` gives a percentile (`latency_p95_s`: 95% of requests were faster than this).

In [ ]:
def safe_mean(values):
    v = pd.to_numeric(pd.Series(list(values), dtype="object"), errors="coerce").dropna()
    return float(v.mean()) if len(v) else None


def pctl(values, p):
    v = pd.to_numeric(pd.Series(list(values), dtype="object"), errors="coerce").dropna()
    return float(np.percentile(v, p)) if len(v) else None

In [ ]:
def summarize(d):
    """One row of metrics for one model's runs (see the A1 metrics table)."""
    ans, una, trap = d[~d["unanswerable"]], d[d["unanswerable"]], d[d["trap"]]
    return {
        "judge_correct": safe_mean(ans["judge_score"]),       # answerable questions, including traps
        "fact_recall": safe_mean(ans["fact_recall"]),
        "trap_accuracy": safe_mean(trap["judge_score"]),      # trap questions only
        "abstention_rate": safe_mean(una["abstained"]),       # declined on UNanswerable (good)
        "false_abstention": safe_mean(ans["abstained"]),      # declined on answerable (bad)
        "format_ok": safe_mean(d["format_ok"]),
        "latency_p50_s": pctl(d["latency_s"], 50),
        "latency_p95_s": pctl(d["latency_s"], 95),
        "out_tokens_mean": safe_mean(d["out_tokens"]),
    }

In [ ]:
comparison = pd.DataFrame({m: summarize(runs[runs["model"] == m]) for m in CANDIDATES}).T
display(comparison.astype(float).round(3))
for m in CANDIDATES:
    for k in ["judge_correct", "trap_accuracy", "abstention_rate", "latency_p50_s"]:
        v = comparison.loc[m, k]
        ck.record(f"A6_{m}_{k}", None if v is None or pd.isna(v) else float(v))

**Reading the table:** one row per model and one column per metric from the A1 table. Shares are between 0 and 1 (0.75 = 75%), latencies are in seconds, and `out_tokens_mean` is tokens per answer. The A1 table says whether higher or lower is better for each column.

**TODO:** read the table like a reviewer would.
1. `BEST_MODEL_ON`: for each of the three metrics, the model that does **best** (check the A1 table: for latency, lower is better). If two models tie, either one is fine.
2. `BIGGEST_TRAP_DROP`: the model whose `trap_accuracy` falls furthest **below** its own `judge_correct`, i.e. the one most thrown off by conflicting documents.

In [ ]:
BEST_MODEL_ON = {"trap_accuracy": "", "abstention_rate": "", "latency_p95_s": ""}   # TODO: a model name for each
BIGGEST_TRAP_DROP = ""    # TODO: the model with the largest gap between judge_correct and trap_accuracy

In [ ]:
def tied_best(values, higher_is_better=True):
    """Models whose value equals the best one (ties allowed). Empty set if there are no numbers."""
    v = {m: float(x) for m, x in values.items() if x is not None and not pd.isna(x)}
    if not v:
        return set()
    best = max(v.values()) if higher_is_better else min(v.values())
    return {m for m, x in v.items() if abs(x - best) < 1e-9}


best_ok = all(BEST_MODEL_ON.get(k) in tied_best(comparison[k].to_dict(), METRICS[k])
              for k in ["trap_accuracy", "abstention_rate", "latency_p95_s"])
drops = {m: (None if pd.isna(comparison.loc[m, "judge_correct"]) or pd.isna(comparison.loc[m, "trap_accuracy"])
             else float(comparison.loc[m, "judge_correct"]) - float(comparison.loc[m, "trap_accuracy"])) for m in CANDIDATES}
ck.check("A6_best_models", best_ok, points=2,
         hint="For each metric, find the best value in its column (highest for shares like trap_accuracy and abstention_rate, "
              "lowest for latency) and copy that model's name exactly.")
ck.check("A6_trap_drop", BIGGEST_TRAP_DROP in tied_best(drops), points=1,
         hint="For each model, compare judge_correct with trap_accuracy in the same row; pick the model where trap_accuracy is lowest relative to it.")
ck.record("A6_best_model_on", BEST_MODEL_ON)
ck.record("A6_biggest_trap_drop", BIGGEST_TRAP_DROP)

## A7. Trap-question breakdown

On the trap questions, the context had two documents that disagree. For each model, how often did it give the **current** answer, how often did it give an answer that **contradicts** the reference (likely following the other document), and how often did it decline?

`trap_breakdown()` (given) counts, for each model's 16 trap answers, the share (a) judged correct, (b) flagged `contradicts` by the judge, and (c) abstained.

In [ ]:
def trap_breakdown(d):
    """Shares of a model's trap answers that were current, contradicted the reference, or abstained."""
    t = d[d["trap"]]
    n = len(t)
    if n == 0:
        return {"current": None, "contradicts_reference": None, "abstained": None, "n": 0}
    return {"current": float((t["verdict"] == "correct").sum() / n),
            "contradicts_reference": float((t["contradicts"] == True).sum() / n),
            "abstained": float(t["abstained"].sum() / n), "n": n}

In [ ]:
traps = pd.DataFrame({m: trap_breakdown(runs[runs["model"] == m]) for m in CANDIDATES}).T
display(traps)
nontrap = {m: safe_mean(runs[(runs["model"] == m) & ~runs["trap"] & ~runs["unanswerable"]]["judge_score"]) for m in CANDIDATES}
print("judge_correct on ordinary (non-trap) answerable questions:", {m: None if v is None else round(v, 2) for m, v in nontrap.items()})
for m in CANDIDATES:
    v = traps.loc[m, "contradicts_reference"]
    ck.record(f"A7_{m}_trap_contradicts", None if v is None or pd.isna(v) else float(v))

**Reading the table:** one row per model, over its `n` = 16 trap answers. `current` is the share the judge marked fully correct (it gave the current answer), `contradicts_reference` the share that stated a different specific fact (usually taken from the outdated document), and `abstained` the share where it declined. They don't have to add up to 1: an answer can be partly correct, or wrong without contradicting anything. The line printed under the table is the judge score on the ordinary (non-trap) questions, for comparison.

Now look at the cases behind the numbers. For each trap question a model got wrong, the cell prints the answer and the **headers** of the documents in its context (doc id, version, effective date). Read two or three of them against the documents. Which document did the model follow? What in the context *could* have told it which one to trust?

In [ ]:
wrong = runs[runs["trap"] & (runs["contradicts"] == True)].sort_values("qid")
print(len(wrong), "trap answers flagged as contradicting the reference\n")
for qid, grp in list(wrong.groupby("qid"))[:6]:
    print(f"{qid}: {BY_ID[qid]['question']}")
    print("   reference:", BY_ID[qid]["reference_answer"])
    for p in CTX[qid]:
        print(f"   context doc: {p['doc_id']} v{p['version']} (effective {p['effective_date']}) - {p['section'][:50]}")
    for _, r in grp.iterrows():
        print(f"   {r['model']:11} said: {r['text'][:160]}")
    print()

**TODO:**
1. `MOST_MISLED`: the model with the highest `contradicts_reference` share in the trap table (ties: either one).
2. `TRAP_FIX`: from the cases above, one concrete change to how the assistant's documents are prepared or presented that would cut these errors, and why. Lab 07 builds the context for the assistant, so this is your first design note for it.

In [ ]:
MOST_MISLED = ""     # TODO: model name with the highest contradicts_reference share
TRAP_FIX = {
    "change": "",    # TODO: one concrete change (a short sentence)
    "why": "",       # TODO: why it would help, based on the cases you read
}

In [ ]:
ck.check("A7_most_misled", MOST_MISLED in tied_best(traps["contradicts_reference"].to_dict()), points=1,
         hint="Look down the contradicts_reference column of the trap table and copy the name of the model with the highest share.")
ck.check("A7_trap_fix", len(str(TRAP_FIX.get("change", "")).split()) >= 5 and len(str(TRAP_FIX.get("why", "")).split()) >= 8, points=1,
         hint='Write a concrete change (5+ words) and a reason based on the printed cases (8+ words).')
ck.record("A7_most_misled", MOST_MISLED)
ck.record("A7_trap_fix", TRAP_FIX)

## A8. Weighted scorecard and your choice

To combine metrics with different units, rescale each one across the four models to 0-1 (best model = 1, worst = 0), flipping metrics where lower is better. Then each model's score is the weighted sum over your `CRITERIA`.

`rescale()` (given) does this: the best of the four models gets 1, the worst gets 0, and the others land in between in proportion to their raw value (flipped where lower is better). If all four have the same value, everyone gets 1.0; a missing value gets 0.

**TODO (before you run the example):** two made-up metrics for the four models are below. Predict which model `rescale()` gives **1.0** and which it gives **0.0** for latency, where lower is better.

In [ ]:
def rescale(values, higher_is_better=True):
    """values: dict model -> number (or None). Returns dict model -> 0..1 (best 1, worst 0)."""
    nums = {m: v for m, v in values.items() if v is not None and not pd.isna(v)}
    if not nums:
        return {m: 0.0 for m in values}
    lo, hi = min(nums.values()), max(nums.values())
    out = {}
    for m, v in values.items():
        if m not in nums:
            out[m] = 0.0
        elif hi == lo:
            out[m] = 1.0
        else:
            s = (v - lo) / (hi - lo)               # 0 at the lowest value, 1 at the highest
            out[m] = s if higher_is_better else 1 - s
    return out


TOY_LATENCY = {"qwen-small": 0.6, "qwen-moe": 1.1, "gemma-moe": 1.3, "qwen-dense": 4.2}       # p50 seconds, lower is better
TOY_CORRECT = {"qwen-small": 0.88, "qwen-moe": 0.91, "gemma-moe": 0.90, "qwen-dense": 0.92}  # judge_correct, higher is better

In [ ]:
RESCALE_PREDICT = {"gets_1": "", "gets_0": ""}    # TODO: model names, for TOY_LATENCY

In [ ]:
toy = pd.DataFrame({"latency_raw": TOY_LATENCY, "latency_rescaled": rescale(TOY_LATENCY, higher_is_better=False),
                    "correct_raw": TOY_CORRECT, "correct_rescaled": rescale(TOY_CORRECT)})
display(toy.round(3))

**Reading the example:** look at `correct_raw`. The best and worst models are only a few points apart, but after rescaling one gets 1.0 and the other 0.0, the same spread as the huge latency difference. With 32 answerable questions, one question is worth about 0.03.

**TODO:** set `TOY_RAW_GAP` to the raw difference between the best and worst `correct_raw` values, read from the table (a number like 0.07). Keep this in mind when you defend your pick in Q1.

In [ ]:
TOY_RAW_GAP = None    # TODO: best minus worst correct_raw, read from the table

In [ ]:
score_rows = {}
for crit, c in CRITERIA.items():
    metric = c.get("metric")
    if metric in METRICS:
        scaled = rescale({m: comparison.loc[m, metric] for m in CANDIDATES}, METRICS[metric])
        score_rows[f"{crit} ({metric})"] = scaled
scorecard = pd.DataFrame(score_rows)
weights = pd.Series({f"{k} ({c.get('metric')})": (c.get("weight") or 0) for k, c in CRITERIA.items() if c.get("metric") in METRICS})
if len(scorecard.columns):
    scorecard["WEIGHTED_TOTAL"] = (scorecard[weights.index] * weights).sum(axis=1)
    display(scorecard.round(3).sort_values("WEIGHTED_TOTAL", ascending=False))

**Reading the scorecard:** one row per model, best first. Each column is one of your criteria (with its metric in brackets), rescaled so the best of the four models gets 1 and the worst gets 0, already flipped where lower is better. `WEIGHTED_TOTAL` multiplies each column by your weight and adds them up. Rescaling stretches small gaps: a 1 versus 0 can hide a tiny real difference, so check the raw numbers in the A6 table before you decide.

**What if your weights were different?** Q1 asks whether a reasonable change in weights would change the winner. Try it here without touching `CRITERIA`: edit `what_if` (for example, double `cost_latency`) and run the cell. The weights are re-scaled to add up to 1 for you. This is optional and not graded.

In [ ]:
what_if = {k: c.get("weight") or 0 for k, c in CRITERIA.items() if c.get("metric") in METRICS}
# what_if["cost_latency"] = what_if["cost_latency"] * 2      # <- example: try a change, then run
if len(scorecard.columns) and sum(what_if.values()) > 0:
    w = pd.Series({f"{k} ({CRITERIA[k]['metric']})": v for k, v in what_if.items()})
    w = w / w.sum()
    print("Weighted total with the what-if weights:")
    print((scorecard[w.index] * w).sum(axis=1).sort_values(ascending=False).round(3).to_string())

In [ ]:
CHOSEN_MODEL = ""   # TODO: the model you pick for the help-desk assistant ("qwen-moe", "qwen-dense", "gemma-moe" or "qwen-small")

In [ ]:
toy_lat = rescale(TOY_LATENCY, higher_is_better=False)
ck.check("A8_rescale_predicted", toy_lat.get(RESCALE_PREDICT.get("gets_1")) == 1.0 and toy_lat.get(RESCALE_PREDICT.get("gets_0")) == 0.0,
         points=1, hint="For latency, lower is better: the fastest model gets 1.0 and the slowest gets 0.0. Check the latency_rescaled column.")
ck.check("A8_raw_gap", isinstance(TOY_RAW_GAP, (int, float)) and abs(TOY_RAW_GAP - (max(TOY_CORRECT.values()) - min(TOY_CORRECT.values()))) < 0.0051,
         points=1, hint="Subtract the smallest correct_raw value from the largest (both are in the table).")
ck.check("A8_model_chosen", CHOSEN_MODEL in CANDIDATES, points=1, hint="Set CHOSEN_MODEL to one of the four model names.")
ck.record("A8_chosen_model", CHOSEN_MODEL)
if "WEIGHTED_TOTAL" in scorecard:
    ck.record("A8_scorecard", {m: round(float(v), 3) for m, v in scorecard["WEIGHTED_TOTAL"].items()})
print("Requests sent this session (cached replies not counted):", REQUESTS["n"])
ck.record("partA_requests_this_session", REQUESTS["n"])

## A9. What would it cost per month?

Your scorecard measured quality and speed. An IT manager's next question is "what will this cost us?" Chapter 4's *Cost and Latency* criterion (under *Evaluation Criteria*) explains the two ways to pay:

- **Model API** (buy): you pay per token, input and output. The cost per token stays about the same as you grow, so the bill grows in a straight line with use.
- **Self-hosting** (build): you pay mostly fixed costs (hardware, power, and engineers' time) whether you send a thousand questions or a million, so the cost *per question* drops as volume grows.

That is why the chapter says companies should re-check this choice at different scales. The **break-even volume** is where the two costs are equal.

The prices below are **pretend** (invented for this exercise; real prices change constantly): API prices per 1 million tokens for models like ours, and a fixed monthly cost for running our own server. The tokens per question come from **your** A4 run.

In [ ]:
PRETEND_API_PRICES = {            # USD per 1M tokens: (input, output). Invented numbers for this exercise.
    "qwen-small": (0.10, 0.40), "qwen-moe": (0.30, 1.20), "gemma-moe": (0.30, 1.20), "qwen-dense": (0.60, 2.40)}
ENGINEER_RATE = 75.0               # USD per hour
SELF_HOSTED_MONTHLY = {"hardware_amortized": 12000 / 36,               # a $12,000 server paid off over 3 years
                       "power": 40.0,
                       "engineer_hours": 6 * ENGINEER_RATE}             # 6 hours a month of upkeep
SELF_HOSTED_TOTAL = sum(SELF_HOSTED_MONTHLY.values())
print(f"Self-hosted fixed cost: ${SELF_HOSTED_TOTAL:,.2f} per month")

# Mean tokens per question from your A4 run. If the gateway did not report token counts,
# they are estimated from text length (about 4 characters per token) so the cost model still works.
est_in = runs["qid"].map(lambda i: sum(len(m["content"]) for m in build_messages(i)) / 4)
est_out = runs["text"].fillna("").str.len() / 4
tok = pd.DataFrame({"model": runs["model"],
                    "in_tokens": pd.to_numeric(runs["in_tokens"], errors="coerce").fillna(est_in),
                    "out_tokens": pd.to_numeric(runs["out_tokens"], errors="coerce").fillna(est_out)})
if runs["in_tokens"].isna().any() or runs["out_tokens"].isna().any():
    print("Note: some token counts were missing, so they were estimated from text length.")
tok_stats = tok.groupby("model")[["in_tokens", "out_tokens"]].mean()
display(tok_stats.round(1))

**Reading the table:** average tokens per question for each model. `in_tokens` is everything sent (system prompt, documents, question) and `out_tokens` is the answer. Input is much larger because every question carries its documents. The models' tokenizers differ (Lab 01), so the same text can count as a different number of tokens.

**TODO (three small ones):**
1. `cost_per_question()`: the dollars for one question. Prices are per **1 million** tokens, so multiply each token count by its price, add them, and divide by 1,000,000.
2. `break_even_volume()`: the questions per month at which the API bill equals `SELF_HOSTED_TOTAL`. Return `None` if the cost per question is missing or zero.
3. `LARKSPUR_Q_PER_MONTH`: your estimate of how many questions Larkspur's assistant would get per month, and one sentence on how you got it. Larkspur has 24 staff and 40 client companies, and the help desk works tickets all day. There is no single right number; a rough estimate with a stated reason is how real cost plans start.

In [ ]:
def cost_per_question(model, prices=PRETEND_API_PRICES):
    """Pretend API cost in USD of ONE question for `model`, using measured mean tokens per question."""
    pin, pout = prices[model]           # USD per 1M input tokens, per 1M output tokens
    tin, tout = tok_stats.loc[model, "in_tokens"], tok_stats.loc[model, "out_tokens"]
    return None   # TODO: input tokens x input price plus output tokens x output price, remembering prices are per 1M tokens


def hosted_monthly_cost(model, questions_per_month, prices=PRETEND_API_PRICES):
    c = cost_per_question(model, prices)
    return None if c is None else c * questions_per_month


def break_even_volume(model, prices=PRETEND_API_PRICES):
    """Questions per month at which the pretend API costs as much as self-hosting (None if undefined)."""
    return None   # TODO: use cost_per_question(model, prices) and SELF_HOSTED_TOTAL; return None if the cost is None or 0


LARKSPUR_Q_PER_MONTH = None   # TODO: your estimate (a number)
VOLUME_REASON = ""            # TODO: one sentence on how you estimated it

In [ ]:
def _num(x):
    return isinstance(x, (int, float, np.integer, np.floating)) and not isinstance(x, bool) and not pd.isna(x)


vol = LARKSPUR_Q_PER_MONTH if _num(LARKSPUR_Q_PER_MONTH) and LARKSPUR_Q_PER_MONTH > 0 else None
VOLUMES = ([int(vol)] if vol else []) + [5_000, 50_000, 500_000]   # questions per month
cost_table = pd.DataFrame({f"{v:,} q/month": {m: hosted_monthly_cost(m, v) for m in CANDIDATES} for v in VOLUMES})
cost_table["self_hosted_per_month"] = SELF_HOSTED_TOTAL
cost_table["break_even_q_per_month"] = pd.Series({m: break_even_volume(m) for m in CANDIDATES})
display(cost_table.astype(float).round(2))

m0 = list(CANDIDATES)[0]
expect = (tok_stats.loc[m0, "in_tokens"] * PRETEND_API_PRICES[m0][0]
          + tok_stats.loc[m0, "out_tokens"] * PRETEND_API_PRICES[m0][1]) / 1e6
per_q = {m: cost_per_question(m) for m in CANDIDATES}
ck.check("A9_cost_per_question", _num(per_q[m0]) and abs(per_q[m0] - expect) < 1e-12
         and all(_num(c) and 0 < c < 0.05 for c in per_q.values()), points=2,
         hint="Prices are dollars per 1,000,000 tokens: combine input and output tokens with their prices, then scale. "
              "With these prices one question costs a small fraction of a cent.")
be = {m: break_even_volume(m) for m in CANDIDATES}
ck.check("A9_break_even", all(_num(x) and x > 0 and _num(per_q[m]) and abs(hosted_monthly_cost(m, x) - SELF_HOSTED_TOTAL) < 0.01 * SELF_HOSTED_TOTAL
                              for m, x in be.items()), points=1,
         hint="Finish cost_per_question() first. At the break-even volume, hosted_monthly_cost(model, volume) should equal SELF_HOSTED_TOTAL.")
ck.check("A9_volume_estimate", vol is not None and 100 <= vol <= 1_000_000 and len(VOLUME_REASON) >= 20, points=1,
         hint="Set LARKSPUR_Q_PER_MONTH to a number between 100 and 1,000,000 and explain it in VOLUME_REASON.")
for m in CANDIDATES:
    ck.record(f"A9_{m}_cost_per_1000_q", round(float(per_q[m]) * 1000, 4) if _num(per_q[m]) else None)
    ck.record(f"A9_{m}_break_even_q_per_month", round(float(be[m])) if _num(be[m]) else None)
ck.record("A9_larkspur_q_per_month", vol)
if vol and CHOSEN_MODEL in CANDIDATES and _num(per_q[CHOSEN_MODEL]):
    chosen_cost = float(hosted_monthly_cost(CHOSEN_MODEL, vol))
    print(f"Your pick ({CHOSEN_MODEL}) at {int(vol):,} questions/month: about ${chosen_cost:,.2f} per month "
          f"through the pretend API, versus ${SELF_HOSTED_TOTAL:,.2f} per month to self-host.")
    ck.record("A9_chosen_monthly_api_cost", round(chosen_cost, 2))

**Reading the cost table:** one row per model. The `q/month` columns are the pretend API bill in dollars per month at that volume (your Larkspur estimate first). `self_hosted_per_month` is the same fixed amount for every model and volume. `break_even_q_per_month` is the volume at which the API bill would reach that fixed cost; it can be larger than every volume column, in which case the API is cheaper at every volume shown. How far is your Larkspur volume from the break-even? Does cost change the pick your scorecard made? Q5 asks about both.

# Part B (optional stretch, up to 10 bonus points): a build-vs-buy recommendation

This is a Part-A-only week, so Part B is optional. It turns A9 into the kind of short recommendation an IT professional writes for a decision-maker. Chapter 4's *Model Build Versus Buy* section says cost is only one of the factors; it also discusses data privacy, functionality, API cost versus engineering cost, and control, access, and transparency (see the chapter's table of pros and cons of model APIs and self-hosting).

You will (B1) check how sensitive the break-even is to your assumptions, (B2) rate the chapter's non-price factors for Larkspur, and (B3) write a one-page memo to Larkspur's CEO, Dana Whitfield. Bonus checks are reported separately from the Part A score.

In [ ]:
BONUS = {}


def bonus_check(name, passed, points, hint=""):
    BONUS[name] = points if passed else 0
    print(("✅" if passed else "❌"), f" BONUS {name} ({BONUS[name]}/{points})", "" if passed else f"\n     -> {hint}")

## B1. Sensitivity check

A **sensitivity check** asks: if one assumption is wrong, does the conclusion change? `run_scenario()` recomputes the costs with some assumptions changed:

- `price_factor`: multiply every API price (2.0 = prices double),
- `input_factor`: multiply the input tokens per question (Lab 08's retrieval puts more documents into each prompt),
- `engineer_hours`: engineer hours per month that self-hosting needs (A9 assumed 6),
- `volume_factor`: multiply your Larkspur volume (for example, if all 40 clients start using the assistant).

**TODO:** finish the three marked lines in `run_scenario()`, then add **one scenario of your own** to `SCENARIOS` (a name and the settings you want to test).

In [ ]:
def run_scenario(model, price_factor=1.0, input_factor=1.0, engineer_hours=6.0, volume_factor=1.0):
    pin, pout = PRETEND_API_PRICES[model]
    tin, tout = tok_stats.loc[model, "in_tokens"], tok_stats.loc[model, "out_tokens"]
    per_q = None         # TODO: cost of one question as in cost_per_question(), but with the input tokens scaled by input_factor and the result scaled by price_factor
    self_hosted = None   # TODO: monthly self-hosted cost: the hardware and power items of SELF_HOSTED_MONTHLY, plus this scenario's engineer_hours paid at ENGINEER_RATE
    volume = (LARKSPUR_Q_PER_MONTH or 0) * volume_factor
    return {"api_monthly": None if per_q is None else per_q * volume, "self_hosted_monthly": self_hosted,
            "break_even_q_per_month": None}   # TODO: the volume where the API bill equals self_hosted (as in break_even_volume); None if per_q is missing or 0


SCENARIOS = {
    "baseline": {},
    "API prices double": {"price_factor": 2.0},
    "API prices halve": {"price_factor": 0.5},
    "5x more context per question (RAG)": {"input_factor": 5.0},
    "self-hosting needs 20 engineer hours/month": {"engineer_hours": 20},
    # TODO: add one scenario of your own, e.g. "name": {"volume_factor": 10}
}

In [ ]:
B_MODEL = CHOSEN_MODEL if CHOSEN_MODEL in CANDIDATES else "qwen-moe"
scen = pd.DataFrame({name: run_scenario(B_MODEL, **kw) for name, kw in SCENARIOS.items()}).T
print(f"Scenarios for {B_MODEL} at {LARKSPUR_Q_PER_MONTH} questions/month (baseline):")
display(scen)
base = run_scenario(B_MODEL)
dbl = run_scenario(B_MODEL, price_factor=2.0)
hrs = run_scenario(B_MODEL, engineer_hours=20)
ok_base = _num(base["break_even_q_per_month"]) and _num(be.get(B_MODEL)) and abs(base["break_even_q_per_month"] - be[B_MODEL]) < 0.01 * be[B_MODEL]
bonus_check("B1_scenario_function", ok_base and _num(base["self_hosted_monthly"]) and abs(base["self_hosted_monthly"] - SELF_HOSTED_TOTAL) < 0.01, 2,
            "With default settings, run_scenario() should reproduce A9's break-even and SELF_HOSTED_TOTAL.")
bonus_check("B1_sensitivity_direction", ok_base and _num(dbl["break_even_q_per_month"])
            and abs(dbl["break_even_q_per_month"] / base["break_even_q_per_month"] - 0.5) < 0.01
            and _num(hrs["self_hosted_monthly"]) and hrs["self_hosted_monthly"] > base["self_hosted_monthly"], 2,
            "Doubling API prices should halve the break-even; more engineer hours should raise the self-hosted cost.")
bonus_check("B1_own_scenario", len(SCENARIOS) >= 6, 1, "Add one scenario of your own to SCENARIOS.")

## B2. The factors besides price

**TODO:** for each of these four factors from the chapter's *Model Build Versus Buy* section (the section discusses a few more), say which option it favors **for Larkspur** (`"api"`, `"self_host"`, or `"neutral"`) and why, in one sentence about Larkspur specifically. Think about what the assistant would see (tickets and documents about 40 client companies), what it needs (Lab 09 adds tool calling), who would run a self-hosted server, and what happens if a vendor changes or retires a model your evaluation was done on.

In [ ]:
FACTORS = {   # TODO: favors = "api", "self_host" or "neutral"; why = one sentence about Larkspur
    "data_privacy":                 {"favors": "", "why": ""},
    "functionality":                {"favors": "", "why": ""},
    "api_cost_vs_engineering_cost": {"favors": "", "why": ""},
    "control_access_transparency":  {"favors": "", "why": ""},
}

In [ ]:
bonus_check("B2_factors", all(f.get("favors") in ("api", "self_host", "neutral") and len(f.get("why", "")) >= 30 for f in FACTORS.values())
            and len(FACTORS) >= 4, 2, "Every factor needs favors = api / self_host / neutral and a 'why' sentence (30+ characters).")

## B3. Memo to the CEO

**TODO:** write a short memo (about 150-250 words) to Dana Whitfield, Larkspur's CEO, recommending **buy** (a paid API), **build** (self-host), or a mix. Write for a busy business owner, not an engineer. Include:

- your recommendation in the first sentence,
- the monthly cost of each option at your volume estimate, and the break-even volume (your numbers from A9),
- what the sensitivity check showed (does any reasonable scenario flip the answer?),
- at least two non-price factors from B2,
- what would make you revisit the decision.

In [ ]:
MEMO = """TODO: your memo to Dana Whitfield (about 150-250 words, using your numbers)."""

In [ ]:
factor_words = ["privacy", "functionality", "function calling", "structured", "engineer", "control", "version", "rate limit",
                "transparen", "access", "sla", "vendor", "lock"]
memo_low = MEMO.lower()
bonus_check("B3_memo", "TODO" not in MEMO and len(MEMO.split()) >= 120
            and sum(w in memo_low for w in factor_words) >= 2, 2,
            "Write at least 120 words and discuss at least two non-price factors (privacy, functionality, engineering effort, control...).")
bonus_check("B3_memo_numbers", len(re.findall(r"\$\s?\d", MEMO)) >= 2 and "break" in memo_low, 1,
            "Use your own numbers: at least two dollar amounts and the break-even volume.")
ck.record("partB_bonus_points", sum(BONUS.values()))
ck.record("partB_bonus_max", 10)

# Written analysis

Use **your numbers**. Cite Chapter 4 by section name where asked. A short paragraph each.

**Q1. Your pick.** Which model did you choose, and what in your comparison table and scorecard supports it? Would a reasonable change in your weights (say, doubling the cost/latency weight) change the winner? Relate your criteria to the chapter's *Evaluation Criteria* (domain-specific capability, generation capability, instruction-following capability, cost and latency).

*Your answer:*

**Q2. Trap questions.** How often did each model follow the outdated document? Describe one case in detail: the two documents, what the model said, and what in the context could have told it which document to trust. Is choosing the document with the latest effective date always enough? Look closely at your trap questions' contexts before answering (the A2 list of trap headers, and `print(format_context("Q..."))` for any question): think about what *kind* of document each one is, not only its date. What would you change in the system prompt or the context (Lab 7) to fix this?

*Your answer:*

**Q3. Benchmarks vs. your own evaluation.** The chapter's *Model Selection Workflow* runs from filtering by hard attributes, to public information, to your own evaluation, to monitoring in production. Which steps did this lab cover, and which did it skip? Explain *Data contamination with public benchmarks* and why it makes leaderboard scores a weak guide here. Then notice that our gold set lives in a **public** GitHub repository: what does that mean for anyone who evaluates future models on it? (Think about where models get their training data: Chapter 2 and Lab 01.)

*Your answer:*

**Q4. How much do you trust these numbers?** Start from your A5b spot check: in how many of your cases did you agree with the judge, and what kind of mistake did it make, if any (for example, going easy on `qwen-moe`'s own answers, as in Lab 04)? Then name one more weakness of this evaluation (think: 40 questions, latency measured on a shared cluster, a regex for abstention, the `fact_recall` failures you found in A3). For each weakness, say how you would fix it before Larkspur commits to a model.

*Your answer:*

**Q5. What would it cost?** Using your A9 numbers, what would your chosen model cost per month through the pretend API at your Larkspur volume, and what is its break-even volume against self-hosting? Explain, citing Chapter 4's *Cost and Latency* criterion, why API cost and self-hosted cost grow differently with volume. Does cost change the pick your scorecard made? Finally, name one cost that the A9 model leaves out or underestimates (the chapter's *Model Build Versus Buy* section, especially *API cost versus engineering cost*, has ideas).

*Your answer:*

# AI-use reflection

1. What did you use AI assistance for in this lab (tools, which steps)?

*Your answer:*

2. Where did the AI help most, and where was it wrong or unhelpful? How did you notice?

*Your answer:*

3. What is one thing you now understand that you could explain without AI help?

*Your answer:*

4. Before you saw the results, which model did you expect to win, and why? Did the numbers change your mind?

*Your answer:*

## Submit

Run this last, save, upload the .ipynb. (Part B bonus points appear in the records as `partB_bonus_points`.)

In [ ]:
ck.report()